# Non-specular RuO2/TiO2 CTRs: where DWBA and the kinematical model disagree

`RuO2_roughness_models_DWBA.ipynb` shows that the specular $(0,0,L)$ rod of the
Poisson-dissolution family agrees with the kinematical calculation except at low
$L$, while the non-specular $(0,1,L)$ rod keeps a visible offset up to the top of
the range. The offset is largest for the heavily dissolved models: near
$(0,1,3)$ with more than four dissolved monolayers the kinematical amplitude is
about a factor two below the DWBA amplitude, far from any Bragg peak and much
larger than the refraction shift in $L$.

This notebook extends the comparison to the $(1,1,L)$ and $(2,0,L)$ rods and
then takes the disagreement apart. The raw comparison mixes four separate
effects, three of which are genuine physics and one of which is a bookkeeping
artefact:

1. the **polarization contraction** $\cos\varphi$, which the DWBA matrix element
   contains and the bare kinematical structure factor `SXRDCrystal.F` does not;
2. the **optical field amplitude** $|A_i^{+}A_f^{+}|$ in the medium that holds
   the scatterers, plus interference with the reflected channels. This is real
   DWBA physics, and here it is dominated by the dense RuO2 film rather than by
   the TiO2 substrate;
3. the **bulk truncation prescription**: the kinematical model damps the bulk
   geometric series with the empirical `SXRDCrystal.atten`, the DWBA damps it
   with the true complex $k_z$. Off Bragg this changes the *bulk* term by about
   one percent, which is invisible on a smooth rod but dominates wherever the
   total amplitude is a near-cancellation, which is exactly the deeply dissolved
   anti-Bragg region;
4. the **refraction shift** of $Q_z$, which is negligible off Bragg and decisive
   at a Bragg pole.

Item 3 is not physics. It is a mismatch between two different truncation
conventions, and `CTRutil.attenuation_from_dwba` removes it.

In [ ]:
%matplotlib widget
from pathlib import Path
import copy
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np


def find_repository_root():
    """Find the checkout containing the current example notebook."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "orgui" / "datautils").is_dir():
            return candidate
    return None


repository_root = find_repository_root()
if repository_root is not None and "orgui" not in sys.modules:
    sys.path.insert(0, str(repository_root))

from orgui.datautils.xrayutils import CTRcalc, CTRoptics, CTRuc, CTRutil
from orgui.datautils.xrayutils.CTRoptics import homogeneous_bulk_profile


def find_example_file(filename):
    """Find an example file from the checkout or notebook directory."""
    candidates = (Path(filename), Path("examples/CTR") / filename)
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(filename)


if not hasattr(CTRcalc.SXRDCrystal, "dwba"):
    raise RuntimeError("The loaded orGUI build does not provide DWBA.")
if not hasattr(CTRuc._CTRcalc_cpp, "unitcell_F_DWBA_records"):
    raise RuntimeError(
        "Rebuild the native extension before running this notebook."
    )

## Model, critical angles, and the incidence angle actually used

The model is the signed-Poisson dissolution model of
`RuO2_roughness_models_DWBA.ipynb`: a semi-infinite TiO2 bulk, an epitaxy
interface, an eleven-cell RuO2 film, and a `PoissonSurface` whose `mean_change`
sets the mean number of dissolved structural layers.

That notebook fixes $\alpha_i=1.5\,\alpha_c$ with $\alpha_c$ taken from the
**TiO2 substrate**. The RuO2 film is considerably denser and therefore has its
own, larger critical angle. Printing both is the first step of the diagnosis:
at $1.5\,\alpha_c(\mathrm{TiO_2})$ the beam is only marginally above the RuO2
critical angle, which is where the transmitted field inside the film peaks.

In [ ]:
model_path = find_example_file("RuO2_TiO2_Poisson_etching.xtal")
model_template = CTRcalc.SXRDCrystal.fromFile(model_path)

delta_bulk = float(
    homogeneous_bulk_profile(model_template.uc_bulk).values[0, 1]
)
alpha_c = float(np.sqrt(2.0 * delta_bulk))
alpha_i_fixed = 1.5 * alpha_c

authored_atten = float(model_template.atten)
bulk_repeat = float(model_template.uc_bulk.a[2])

print(f"energy:              {model_template.uc_bulk._E * 1e-3:g} keV")
print(f"bulk repeat c:       {bulk_repeat:.4f} Angstrom")
print(f"serialized atten:    {authored_atten:g} per bulk repeat")
print(f"alpha_c (TiO2 bulk): {np.rad2deg(alpha_c):.5f} deg")
print(f"fixed alpha_i:       {np.rad2deg(alpha_i_fixed):.5f} deg")

## Rod evaluation with three matched references

`evaluate_rod` returns, for one rod:

* `F_kinematic` -- `SXRDCrystal.F` with the serialized `atten`, that is, exactly
  the dashed curves of the reference notebook;
* `F_dwba` -- the DWBA contrast structure factor `DWBAResult.F_contrast` at
  fixed $\alpha_i$, which is directly comparable to `SXRDCrystal.F` off
  specular;
* `F_matched` -- `SXRDCrystal.F` re-evaluated point by point with
  `CTRutil.attenuation_from_dwba(crystal, alpha_i, alpha_f)`, the kinematical
  attenuation exponent that the DWBA wavevectors actually represent for that
  angle pair. `SXRDCrystal.atten` is one scalar model parameter, so matching it
  to a rod whose exit angle varies with $L$ needs a loop;
* `F_shifted` -- the same kinematical rod evaluated at the refraction-corrected
  $l$, using `refraction_L_shift`. The internal normal wavevectors are smaller
  than the vacuum ones, so a DWBA feature sits at slightly larger nominal $l$;
* `cos_phi` -- `PreparedCTR.cos_azimuth`, the $\sigma$-polarization contraction
  carried by the DWBA matrix element.

`refraction_L_shift` takes the medium as an argument, because the shift is not
one number for the whole rod: the Bragg poles are carried by the semi-infinite
substrate, while the rod between them is carried by the film, whose $\delta$ is
larger.

Amplitudes are in electrons per configured reference lateral cell. Angles are
glancing angles in radians.

In [ ]:
def build_poisson_model(dissolved_layers, atten=None):
    """Return a Poisson dissolution model with the given mean loss.

    :param float dissolved_layers:
        Mean number of dissolved structural layers, in layers.
    :param float atten:
        Optional override of the dimensionless bulk attenuation exponent per
        reference-cell out-of-plane repeat. ``None`` keeps the serialized
        value.
    :returns: Configured crystal.
    :rtype: CTRcalc.SXRDCrystal
    """
    model = copy.deepcopy(model_template)
    if atten is not None:
        model.atten = float(atten)
    surface = model["RuO2surface"]
    surface.profile = CTRcalc.PoissonProfile(
        mean_change=-dissolved_layers, alpha=surface.basis[1], offset=0.0
    )
    surface.basis[:] = (
        surface.profile.mean_change, surface.profile.alpha,
        surface.profile.offset,
    )
    surface.basis_0[:] = surface.basis
    model.apply_stacking()
    return model


def refraction_L_shift(prepared, medium="dense"):
    """Return the refraction shift of the nominal ``l`` in r.l.u.

    The internal normal wavevectors are smaller than the vacuum ones, so a
    DWBA feature sits at a slightly larger nominal ``l`` than its kinematical
    counterpart. The shift depends on which medium carries the scatterers,
    so the bulk-dominated Bragg poles follow the substrate while the
    film-dominated rod between them follows the densest medium.

    :param CTRdwba.PreparedCTR prepared: Prepared DWBA geometry.
    :param str medium:
        ``"dense"`` for the densest slab or ``"substrate"`` for the
        terminal one.
    :returns: Shift in reference-frame ``l``, in r.l.u.
    :rtype: numpy.ndarray
    """
    field_i = prepared.field_i
    field_f = prepared.field_f
    if medium == "dense":
        slab = int(np.argmax(1.0 - np.real(np.asarray(field_i.n))))
    elif medium == "substrate":
        slab = len(field_i.n) - 1
    else:
        raise ValueError("medium must be 'dense' or 'substrate'.")
    kz_i = np.abs(np.real(np.asarray(field_i.kz)[slab].ravel()))[
        np.asarray(prepared.field_i_index).ravel()
    ]
    kz_f = np.abs(np.real(np.asarray(field_f.kz)[slab].ravel()))[
        np.asarray(prepared.field_f_index).ravel()
    ]
    k0 = float(prepared.k0)
    vacuum = k0 * (
        np.sin(np.asarray(prepared.alpha_i, dtype=np.float64))
        + np.sin(np.asarray(prepared.alpha_f, dtype=np.float64))
    )
    return bulk_repeat * (vacuum - kz_i - kz_f) / (2.0 * np.pi)


def evaluate_rod(model, rod, L, alpha_i, matched=True, shifted=False):
    """Evaluate one non-specular rod kinematically and in DWBA.

    :param CTRcalc.SXRDCrystal model: Crystal to evaluate.
    :param tuple rod: Reference-frame ``(h, k)`` in r.l.u.
    :param numpy.ndarray L: Reference-frame ``l`` values in r.l.u.
    :param float alpha_i: Incident glancing angle in radians.
    :param bool matched:
        Also evaluate the kinematical rod with the DWBA-equivalent
        attenuation. This costs one ``F`` call per point.
    :param bool shifted:
        Additionally evaluate the matched kinematical rod at the
        refraction-corrected ``l``. This costs a second ``F`` call per point.
    :returns:
        Dictionary of complex amplitudes in electrons plus the optical
        diagnostics used below.
    :rtype: dict
    """
    h = np.full_like(L, rod[0])
    k = np.full_like(L, rod[1])
    stored_atten = model.atten
    F_kinematic = model.F(h, k, L)

    model.dwba.set_ctr_geometry(alpha_i=alpha_i, rods=[rod])
    result = model.dwba.evaluate(h, k, L)
    prepared = result.prepared
    if np.any(prepared.is_specular):
        raise ValueError("evaluate_rod expects a non-specular rod.")

    alpha_f = np.asarray(prepared.alpha_f, dtype=np.float64)
    atten_dwba = CTRutil.attenuation_from_dwba(model, alpha_i, alpha_f)
    L_shift = refraction_L_shift(prepared)
    F_matched = None
    F_shifted = None
    if matched or shifted:
        F_matched = np.empty_like(L, dtype=np.complex128)
        F_shifted = np.empty_like(L, dtype=np.complex128)
        for index in range(L.size):
            model.atten = float(atten_dwba[index])
            single = slice(index, index + 1)
            F_matched[index] = model.F(h[single], k[single], L[single])[0]
            if shifted:
                F_shifted[index] = model.F(
                    h[single], k[single], L[single] - L_shift[index]
                )[0]
        model.atten = stored_atten
    if not shifted:
        F_shifted = None

    return {
        "F_kinematic": F_kinematic,
        "F_dwba": np.asarray(result.F_contrast),
        "F_matched": F_matched,
        "F_shifted": F_shifted,
        "cos_phi": np.asarray(prepared.cos_azimuth, dtype=np.float64),
        "alpha_f": alpha_f,
        "atten_dwba": np.asarray(atten_dwba, dtype=np.float64),
        "L_shift": L_shift,
        "prepared": prepared,
    }

## The rod survey

Three non-specular rods are evaluated for a mean dissolution of zero to six
layers. $(0,1,L)$ reproduces the reference notebook; $(2,0,L)$ and $(1,1,L)$ are
new. The rods differ in where their bulk Bragg poles sit: $(0,1,L)$ and
$(2,0,L)$ peak at even $L$, $(1,1,L)$ at odd $L$. $L=3$ is therefore an
anti-Bragg position on two of the three rods and a Bragg pole on the third.

In [ ]:
rods = ((0.0, 1.0), (1.0, 1.0), (2.0, 0.0))
dissolved_layers = np.arange(0, 7, dtype=float)
L_rod = np.linspace(0.30, 6.0, 761)

survey = {}
for rod in rods:
    survey[rod] = [
        evaluate_rod(build_poisson_model(dissolved), rod, L_rod, alpha_i_fixed)
        for dissolved in dissolved_layers
    ]

reference = survey[rods[0]][0]
print(f"evaluated {len(rods) * dissolved_layers.size} rods")
print(
    "DWBA-equivalent atten spans "
    f"{reference['atten_dwba'].min():.2e} to "
    f"{reference['atten_dwba'].max():.2e}, "
    f"against the serialized {authored_atten:g}"
)
print(
    "cos(azimuth) spans "
    f"{reference['cos_phi'].min():.4f} to {reference['cos_phi'].max():.4f}"
)

In [ ]:
cmap = mpl.colormaps["turbo"]
stack_factor = 10.0
norm = mpl.colors.Normalize(
    vmin=dissolved_layers.min(), vmax=dissolved_layers.max()
)


def rod_label(rod):
    """Return a display label for a reference-frame rod."""
    return f"({rod[0]:g}, {rod[1]:g}, L)"


figure, axes = plt.subplots(
    1, 3, figsize=(15.0, 5.4), constrained_layout=True, sharex=True
)
for axis, rod in zip(axes, rods):
    for plot_index, (dissolved, result) in enumerate(
        zip(dissolved_layers, survey[rod])
    ):
        color = cmap(norm(dissolved))
        scale = stack_factor**plot_index
        axis.semilogy(
            L_rod, np.abs(result["F_kinematic"]) * scale,
            "--", color=color, linewidth=1.0,
        )
        axis.semilogy(
            L_rod, np.abs(result["F_dwba"]) * scale,
            color=color, linewidth=1.25,
        )
    axis.set_title(rod_label(rod))
    axis.set_xlabel(r"$L$ / r.l.u.")
    axis.grid(alpha=0.18, which="both")
axes[0].set_ylabel(r"stacked $|F|$ / electrons")
axes[0].legend(
    handles=[
        Line2D([0], [0], color="0.25", label=r"DWBA, $\alpha_i=1.5\alpha_c$"),
        Line2D(
            [0], [0], color="0.25", linestyle="--",
            label="kinematical, serialized atten",
        ),
    ],
    loc="lower left", fontsize=8,
)
colorbar = figure.colorbar(
    mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, pad=0.015, shrink=0.86
)
colorbar.set_label("mean dissolved layers")
figure.suptitle(
    "Poisson dissolution: DWBA versus kinematical non-specular CTRs"
)
figure

The mismatch is present on all three rods, so it is not specific to
$(0,1,L)$. It is not a uniform scale factor either: the curves separate most
where the kinematical rod dips into a deep minimum, and those minima sit at
different $L$ on each rod and move with the amount of dissolved material. It is
largest on $(0,1,L)$ between $L=2$ and $L=4$, which is where that rod develops
its deepest minima and is the region the reference notebook reports; the same
signature appears on $(1,1,L)$ near $L=2$ and $L=4$ and on $(2,0,L)$ near
$L=1$, $3$, and $5$, that is, always halfway between the Bragg poles of the rod
in question.

Plotting the amplitude ratio makes the structure explicit.

In [ ]:
figure, axes = plt.subplots(
    1, 3, figsize=(15.0, 4.6), constrained_layout=True, sharex=True, sharey=True
)
for axis, rod in zip(axes, rods):
    for dissolved, result in zip(dissolved_layers, survey[rod]):
        axis.semilogy(
            L_rod,
            np.abs(result["F_dwba"]) / np.abs(result["F_kinematic"]),
            color=cmap(norm(dissolved)), linewidth=1.1,
        )
    axis.axhline(1.0, color="0.35", linewidth=0.9, linestyle=":")
    axis.set_title(rod_label(rod))
    axis.set_xlabel(r"$L$ / r.l.u.")
    axis.grid(alpha=0.18, which="both")
axes[0].set_ylabel(r"$|F_{\mathrm{DWBA}}| / |F_{\mathrm{kin}}|$")
axes[0].set_ylim(0.05, 30.0)
colorbar = figure.colorbar(
    mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, pad=0.015, shrink=0.86
)
colorbar.set_label("mean dissolved layers")
figure.suptitle("Raw amplitude ratio, DWBA over kinematical")
figure

Two features stand out. The narrow **dips to about 0.17** sit exactly on
the bulk Bragg poles: $L=2,4,6$ on $(0,1,L)$ and $(2,0,L)$, $L=1,3,5$ on
$(1,1,L)$. The broad **excursions above two** appear only once material is
dissolved, and they track the kinematical minima. Neither is a constant offset,
so neither can be absorbed into a scale factor during a fit.

## Term 1 and 2: what the DWBA matrix element carries that `F` does not

Two of the four terms are properties of the DWBA matrix element rather than of
the structure, and both can be read straight off the prepared geometry.

`PreparedCTR.cos_azimuth` is the $\sigma$-polarization contraction
$P_{ss}=\cos\varphi$ of the four-channel matrix element. `SXRDCrystal.F` returns
a bare structure factor with no polarization factor, so the DWBA amplitude is
smaller by $\cos\varphi$ at otherwise identical conditions. For these rods
$\varphi$ is the in-plane scattering angle, of order $12^{\circ}$, so
$\cos\varphi \approx 0.97$ to $0.99$.

`field_i.A_plus` and `field_f.A_plus` are the downward electric-field branch
amplitudes in each optical medium. The scatterers that carry the non-specular
amplitude sit in the RuO2 film, not in the ambient, so the relevant enhancement
is $|A_i^{+}A_f^{+}|$ evaluated in the *densest* medium.

In [ ]:
def medium_critical_angles(field):
    """Return the per-medium critical angles of a layered field in radians.

    :param CTRoptics.LayeredElectricField field:
        Field whose slab refractive indices are used.
    :returns: Critical angle per slab, ambient first, in radians.
    :rtype: numpy.ndarray
    """
    delta = 1.0 - np.real(np.asarray(field.n))
    return np.sqrt(np.maximum(2.0 * delta, 0.0))


def dense_medium_transmission(prepared):
    """Return ``|A_i^+ A_f^+|`` in the densest optical medium.

    This is the transmitted-transmitted channel weight that multiplies the
    scatterers carrying the non-specular amplitude.

    :param CTRdwba.PreparedCTR prepared: Prepared DWBA geometry.
    :returns: Dimensionless amplitude weight per evaluated point.
    :rtype: numpy.ndarray
    """
    field_i = prepared.field_i
    field_f = prepared.field_f
    delta = 1.0 - np.real(np.asarray(field_i.n))
    dense = int(np.argmax(delta))
    A_i = np.asarray(field_i.A_plus)[dense].ravel()[
        np.asarray(prepared.field_i_index).ravel()
    ]
    A_f = np.asarray(field_f.A_plus)[dense].ravel()[
        np.asarray(prepared.field_f_index).ravel()
    ]
    return np.abs(A_i * A_f)


# Build the reference model from the unstacked template: an already stacked
# crystal must not be deep-copied again.
pristine_model = build_poisson_model(0.0)
pristine_field = pristine_model.wavefield(alpha_i_fixed)
medium_alpha_c = medium_critical_angles(pristine_field)
alpha_c_film = float(medium_alpha_c.max())

print(f"alpha_c (TiO2 bulk):     {np.rad2deg(alpha_c):.5f} deg")
print(f"alpha_c (densest RuO2):  {np.rad2deg(alpha_c_film):.5f} deg")
print(f"fixed alpha_i:           {np.rad2deg(alpha_i_fixed):.5f} deg")
print(
    f"alpha_i / alpha_c(film): {alpha_i_fixed / alpha_c_film:.3f}"
    "  -- only just above the film critical angle"
)
print(
    "|A_i+ A_f+| in the densest medium: "
    f"{dense_medium_transmission(reference['prepared'])[0]:.4f}"
)

## Term 3: the two bulk truncation prescriptions

The kinematical bulk rod is the geometric series

$$F_{\rm bulk}(l)=\frac{F_{\rm uc}(l)}
{1-\exp(-2\pi i\,l_{\rm bulk}-\mathrm{atten})},$$

and this model file carries `atten = 0.01`. The DWBA does not use that number:
its bulk series is damped by the imaginary part of the internal $k_z$, which at
these angles and $20\;\mathrm{keV}$ corresponds to an equivalent exponent near
$6\times10^{-4}$, roughly seventeen times weaker.

Away from Bragg the denominator has modulus of order one, so the two
prescriptions differ by about one percent *of the bulk term*. On a smooth rod
the bulk term is also the total, and one percent is invisible. On a heavily
dissolved rod the bulk term and the surface correction nearly cancel, and one
percent of the bulk term is then comparable to the whole remaining amplitude.

`CTRutil.attenuation_from_dwba` converts the DWBA wavevectors into the
equivalent scalar kinematical exponent, which makes the comparison fair.

In [ ]:
probe_rod = (0.0, 1.0)
probe_dissolved = 4.0
probe_L = np.array([3.0])
probe_h = np.full_like(probe_L, probe_rod[0])
probe_k = np.full_like(probe_L, probe_rod[1])

probe_model = build_poisson_model(probe_dissolved)
probe = evaluate_rod(probe_model, probe_rod, probe_L, alpha_i_fixed)

serialized = abs(probe["F_kinematic"][0])
matched = abs(probe["F_matched"][0])
dwba = abs(probe["F_dwba"][0])
cos_phi = probe["cos_phi"][0]
transmission = dense_medium_transmission(probe["prepared"])[0]

print(f"rod {rod_label(probe_rod)} at L = {probe_L[0]:g}, "
      f"{probe_dissolved:g} dissolved layers")
print(f"  serialized atten            {authored_atten:.5f}")
print(f"  DWBA-equivalent atten       {probe['atten_dwba'][0]:.5f}")
print(f"  |F| kinematical, serialized {serialized:.5f} electrons")
print(f"  |F| kinematical, matched    {matched:.5f} electrons")
print(f"  |F| DWBA                    {dwba:.5f} electrons")
print()
print(f"  raw ratio                   {dwba / serialized:.3f}")
print(f"  attenuation factor alone    {matched / serialized:.3f}")
print(f"  cos(azimuth)                {cos_phi:.4f}")
print(f"  |A_i+ A_f+| dense medium    {transmission:.4f}")
print(
    "  product of the three        "
    f"{matched / serialized * cos_phi * transmission:.3f}"
)

Almost all of the factor of about two at $(0,1,3)$ is the attenuation
bookkeeping. The genuine optical weight contributes a further factor near
$1.1$ to $1.3$, and the polarization factor pulls back a couple of percent.

The same decomposition over the whole $L$ range, for all three rods, isolates
what is left once the truncation mismatch is removed.

In [ ]:
decomposition_dissolved = 4.0
decomposition = {
    rod: evaluate_rod(
        build_poisson_model(decomposition_dissolved), rod, L_rod,
        alpha_i_fixed, shifted=True,
    )
    for rod in rods
}
print(
    "refraction shift in the densest medium: "
    f"{decomposition[rods[0]]['L_shift'].mean():.4f} r.l.u."
)

figure, axes = plt.subplots(
    1, 3, figsize=(15.0, 4.8), constrained_layout=True, sharex=True, sharey=True
)
for axis, rod in zip(axes, rods):
    result = decomposition[rod]
    dwba_amplitude = np.abs(result["F_dwba"])
    raw = dwba_amplitude / np.abs(result["F_kinematic"])
    matched_ratio = dwba_amplitude / np.abs(result["F_matched"])
    corrected = matched_ratio / result["cos_phi"]
    axis.semilogy(L_rod, raw, color="0.55", linewidth=1.0,
                  label="raw, serialized atten")
    axis.semilogy(L_rod, matched_ratio, color="tab:orange", linewidth=1.1,
                  label="matched atten")
    axis.semilogy(L_rod, corrected, color="tab:blue", linewidth=1.0,
                  label=r"matched atten, $/\cos\varphi$")
    axis.semilogy(
        L_rod,
        dwba_amplitude / (np.abs(result["F_shifted"]) * result["cos_phi"]),
        color="tab:green", linewidth=1.5,
        label=r"matched atten, $/\cos\varphi$, $L$ shifted",
    )
    axis.semilogy(
        L_rod, dense_medium_transmission(result["prepared"]),
        color="tab:red", linewidth=1.1, linestyle="--",
        label=r"$|A_i^+A_f^+|$, dense medium",
    )
    axis.axhline(1.0, color="0.35", linewidth=0.9, linestyle=":")
    axis.set_title(rod_label(rod))
    axis.set_xlabel(r"$L$ / r.l.u.")
    axis.grid(alpha=0.18, which="both")
axes[0].set_ylabel(r"$|F_{\mathrm{DWBA}}| / |F_{\mathrm{kin}}|$")
axes[0].set_ylim(0.08, 12.0)
axes[0].legend(loc="upper left", fontsize=8)
figure.suptitle(
    f"Ratio decomposition at {decomposition_dissolved:g} dissolved layers, "
    rf"$\alpha_i = 1.5\,\alpha_c$"
)
figure

Matching the attenuation removes the broad excursions. What remains are
narrow spikes, and those are term 4 acting off Bragg: at a sharp rod minimum
$|\mathrm{d}F/\mathrm{d}L|$ is large compared with $|F|$, so a shift of only
$0.014$ r.l.u. changes the amplitude by a large *relative* factor even though
the absolute change is tiny. Evaluating the kinematical rod at the
refraction-corrected $l$ (green) removes most of them, and what is left between
them is a smooth factor of about $1.05$ to $1.4$ that follows $|A_i^{+}A_f^{+}|$.
The correction is only approximate at the sharpest minima, because one scalar
shift stands in for records that sit in different media; the leftover spike at
$L=3$ on $(1,1,L)$ is the bulk Bragg pole, which needs the substrate shift
rather than the film one. The scatter around the plateau is interference with
the two reflected channels, whose amplitude $|A_i^{-}|$ in the film is about $0.17$
at this incidence angle.

The case the reference notebook reports is easiest to read as amplitudes rather
than as a ratio.

In [ ]:
zoom_L = np.linspace(2.30, 3.70, 561)
zoom_rod = (0.0, 1.0)
zoom_dissolved = (4.0, 6.0)

figure, axes = plt.subplots(
    1, 2, figsize=(11.5, 4.6), constrained_layout=True, sharex=True
)
for axis, dissolved in zip(axes, zoom_dissolved):
    result = evaluate_rod(
        build_poisson_model(dissolved), zoom_rod, zoom_L, alpha_i_fixed,
        shifted=True,
    )
    axis.semilogy(
        zoom_L, np.abs(result["F_kinematic"]), color="0.55", linestyle="--",
        linewidth=1.1, label="kinematical, serialized atten",
    )
    axis.semilogy(
        zoom_L, np.abs(result["F_shifted"]) * result["cos_phi"],
        color="tab:green", linewidth=1.4,
        label=r"kinematical, matched atten, $L$ shifted, $\times\cos\varphi$",
    )
    axis.semilogy(
        zoom_L, np.abs(result["F_dwba"]), color="tab:blue", linewidth=1.2,
        label=r"DWBA, $\alpha_i = 1.5\,\alpha_c$",
    )
    axis.axvline(3.0, color="0.35", linewidth=0.9, linestyle=":")
    axis.set_title(f"{dissolved:g} dissolved layers")
    axis.set_xlabel(r"$L$ / r.l.u.")
    axis.grid(alpha=0.18, which="both")
axes[0].set_ylabel(r"$|F|$ / electrons")
axes[0].legend(loc="upper left", fontsize=8)
figure.suptitle(r"$(0, 1, L)$ around the reported $(0, 1, 3)$ discrepancy")
figure

The grey curve sits well below the DWBA across the whole window, which is
the reported discrepancy. The green curve is the *same kinematical structure
model*, with only the attenuation convention, the polarization factor and the
refraction shift brought into line, and it tracks the DWBA to within the
residual optical factor. Nothing about the structure changed.

## Born-limit check: does the DWBA reduce to the kinematical rod?

If the implementation is consistent, raising $\alpha_i$ well above every
critical angle in the stack must drive the corrected ratio to one: the field
amplitudes go to unity, the reflected channels vanish, and the refraction shift
goes to zero. The test is run at three anti-Bragg positions on $(0,1,L)$.

In [ ]:
born_multipliers = np.array([1.5, 2.0, 3.0, 5.0, 8.0, 12.0, 20.0])
born_L = np.array([3.15, 4.60, 5.29])
born_model = build_poisson_model(4.0)

born_raw = np.empty((born_multipliers.size, born_L.size))
born_corrected = np.empty_like(born_raw)
born_transmission = np.empty_like(born_raw)
for row, multiplier in enumerate(born_multipliers):
    result = evaluate_rod(
        born_model, (0.0, 1.0), born_L, multiplier * alpha_c, shifted=True
    )
    dwba_amplitude = np.abs(result["F_dwba"])
    born_raw[row] = dwba_amplitude / np.abs(result["F_kinematic"])
    born_corrected[row] = dwba_amplitude / (
        np.abs(result["F_shifted"]) * result["cos_phi"]
    )
    born_transmission[row] = dense_medium_transmission(result["prepared"])

figure, axes = plt.subplots(
    1, 2, figsize=(11.0, 4.4), constrained_layout=True, sharex=True
)
for column, L_value in enumerate(born_L):
    color = mpl.colormaps["viridis"](column / max(born_L.size - 1, 1))
    axes[0].semilogx(
        born_multipliers, born_raw[:, column], "o-", color=color,
        label=f"L = {L_value:g}",
    )
    axes[1].semilogx(
        born_multipliers, born_corrected[:, column], "o-", color=color,
        label=f"L = {L_value:g}",
    )
axes[1].semilogx(
    born_multipliers, born_transmission[:, 0], "k--", linewidth=1.0,
    label=r"$|A_i^+A_f^+|$",
)
axes[0].set_title("raw ratio, serialized atten")
axes[1].set_title(r"matched atten and $L$ shift, $/\cos\varphi$")
for axis in axes:
    axis.axhline(1.0, color="0.35", linewidth=0.9, linestyle=":")
    axis.set_xlabel(r"$\alpha_i / \alpha_c(\mathrm{TiO_2})$")
    axis.set_ylabel(r"$|F_{\mathrm{DWBA}}| / |F_{\mathrm{kin}}|$")
    axis.grid(alpha=0.18, which="both")
    axis.legend(fontsize=8)
figure.suptitle(
    "Born limit of the (0, 1, L) rod at 4 dissolved layers"
)
figure

The corrected ratio approaches one from above as $\alpha_i$ is raised,
tracking $|A_i^{+}A_f^{+}|$ down to within a few percent of unity: with the attenuation matched, the
polarization divided out and the refraction shift removed, the DWBA rod *is* the
kinematical rod in the Born limit. The raw ratio does not converge. It keeps a
residual set by the attenuation mismatch, which does not depend on the incidence
angle. That separation is the cleanest evidence that the persistent high-$L$
offset seen in the reference notebook is a truncation-convention artefact, not a
distorted-wave effect.

## The physical part: the field inside the RuO2 film

The genuine DWBA enhancement is entirely an incident-channel effect at these
angles, because the exit angle on these rods is $13^{\circ}$ or more and the
exit field is already at its vacuum value. Sampling $E(z)$ shows why
$1.5\,\alpha_c(\mathrm{TiO_2})$ is not a weak-optics condition: the RuO2 film
has its own critical angle, close to the chosen $\alpha_i$.

In [ ]:
z_field = np.linspace(20.0, -35.0, 1101)
field_multipliers = np.array([1.0, 1.25, 1.5, 2.0, 3.0, 5.0])
sampled_field = CTRoptics.sample_electric_field(
    pristine_model.wavefield(field_multipliers * alpha_c), z_field
)

delta_profile = 1.0 - np.real(np.asarray(pristine_field.n))
profile_edges = np.asarray(pristine_field.z_interfaces, dtype=np.float64)
dense_slab = int(np.argmax(delta_profile))

profile_z = []
profile_delta = []
for slab, delta_value in enumerate(delta_profile):
    z_high = z_field.max() if slab == 0 else profile_edges[slab - 1]
    z_low = (
        profile_edges[slab] if slab < profile_edges.size else z_field.min()
    )
    profile_z.extend((z_high, z_low))
    profile_delta.extend((delta_value, delta_value))

sweep_multipliers = np.linspace(0.4, 4.0, 361)
sweep_field = pristine_model.wavefield(sweep_multipliers * alpha_c)
sweep_A_plus = np.abs(np.asarray(sweep_field.A_plus)[dense_slab])
sweep_A_minus = np.abs(np.asarray(sweep_field.A_minus)[dense_slab])

figure, axes = plt.subplots(
    1, 3, figsize=(14.5, 4.8), constrained_layout=True
)
axes[0].plot(
    np.asarray(profile_delta) * 1e6, np.asarray(profile_z),
    color="0.25", linewidth=1.2,
)
axes[0].set_xlabel(r"$\delta \times 10^{6}$")
axes[0].set_ylabel(r"$z$ / Angstrom")
axes[0].set_title("optical reference profile")
axes[0].set_ylim(z_field.min(), z_field.max())
axes[0].grid(alpha=0.18)

for column, multiplier in enumerate(field_multipliers):
    color = mpl.colormaps["plasma"](
        0.85 * column / max(field_multipliers.size - 1, 1)
    )
    axes[1].plot(
        np.abs(sampled_field[:, column]) ** 2, z_field, color=color,
        linewidth=1.2, label=rf"{multiplier:g}$\,\alpha_c$",
    )
axes[1].axvline(1.0, color="0.35", linewidth=0.9, linestyle=":")
axes[1].set_ylim(z_field.min(), z_field.max())
axes[1].set_xlabel(r"$|E(z)|^2$ / incident intensity")
axes[1].set_ylabel(r"$z$ / Angstrom")
axes[1].set_title("incident field intensity")
axes[1].grid(alpha=0.18)
axes[1].legend(fontsize=8, title=r"$\alpha_i$", title_fontsize=8)

axes[2].plot(
    sweep_multipliers, sweep_A_plus, color="tab:blue", linewidth=1.4,
    label=r"$|A^{+}|$ transmitted",
)
axes[2].plot(
    sweep_multipliers, sweep_A_minus, color="tab:orange", linewidth=1.4,
    label=r"$|A^{-}|$ reflected",
)
axes[2].axvline(
    1.0, color="0.45", linewidth=1.0, linestyle="--",
    label=r"$\alpha_c(\mathrm{TiO_2})$",
)
axes[2].axvline(
    alpha_c_film / alpha_c, color="tab:red", linewidth=1.0, linestyle="--",
    label=r"$\alpha_c(\mathrm{RuO_2})$",
)
axes[2].axvline(
    1.5, color="0.15", linewidth=1.0, linestyle=":",
    label=r"$\alpha_i$ used here",
)
axes[2].set_xlabel(r"$\alpha_i / \alpha_c(\mathrm{TiO_2})$")
axes[2].set_ylabel("branch amplitude in the densest medium")
axes[2].set_title("field branches inside the RuO2 film")
axes[2].grid(alpha=0.18)
axes[2].legend(fontsize=8)

figure.suptitle(
    rf"$\alpha_c(\mathrm{{TiO_2}})={np.rad2deg(alpha_c):.4f}^\circ$, "
    rf"$\alpha_c(\mathrm{{RuO_2}})={np.rad2deg(alpha_c_film):.4f}^\circ$"
)
figure

The right panel is the point. The transmitted branch $|A^{+}|$ inside the
densest RuO2 medium peaks at $\alpha_c(\mathrm{RuO_2})$, where it reaches about
$3.6$, and the reflected branch $|A^{-}|$ peaks with it. At the chosen
$\alpha_i=1.5\,\alpha_c(\mathrm{TiO_2})=1.21\,\alpha_c(\mathrm{RuO_2})$ the
incidence angle is still on the falling flank of that resonance:
$|A^{+}|\approx1.28$ and $|A^{-}|\approx0.17$. Every scatterer in the film, in
the interface, and in the surface correction is weighted by that field, which is
the physical content of the residual $1.05$ to $1.4$ amplitude factor, and the
reflected branch is large enough to interfere visibly with it.

Two practical consequences follow. Using the substrate critical angle to choose
a "safely kinematical" incidence angle is misleading whenever the film is denser
than the substrate; $3\,\alpha_c(\mathrm{TiO_2})$ would already put $|A^{+}|$
within four percent of unity. And the enhancement is not a rod-independent
constant, because it multiplies the film and surface records differently from
the deeply buried bulk.

## Term 4: refraction at the Bragg poles

The narrow dips to about $0.17$ are a different mechanism. The internal $Q_z$ is
smaller than the nominal one, so the DWBA Bragg maximum sits at slightly higher
nominal $L$. Evaluating both models at the same nominal $L$ therefore compares
the top of one pole with the flank of the other. The shift predicted from the
refracted incident wavevector is

$$\Delta L = \frac{c}{2\pi}
\left(k_{z,\rm vac} - \mathrm{Re}\,k_{z,\rm int}\right),
\qquad
k_{z,\rm int} = k_0\sqrt{\sin^2\alpha_i - \sin^2\alpha_c}.$$

In [ ]:
bragg_L = np.linspace(1.93, 2.09, 641)
bragg_model = build_poisson_model(0.0)
bragg_h = np.zeros_like(bragg_L)
bragg_k = np.ones_like(bragg_L)
bragg_multipliers = (1.5, 2.0, 5.0)

figure, axis = plt.subplots(figsize=(7.6, 4.8), constrained_layout=True)
kinematic_bragg = evaluate_rod(
    bragg_model, (0.0, 1.0), bragg_L, alpha_i_fixed
)
axis.semilogy(
    bragg_L, np.abs(kinematic_bragg["F_matched"]), "k--", linewidth=1.1,
    label="kinematical, matched atten",
)
for column, multiplier in enumerate(bragg_multipliers):
    result = evaluate_rod(
        bragg_model, (0.0, 1.0), bragg_L, multiplier * alpha_c, matched=False
    )
    amplitude = np.abs(result["F_dwba"])
    peak_L = bragg_L[int(np.argmax(amplitude))]
    prepared = result["prepared"]
    k0_value = float(prepared.k0)
    alpha_i_value = multiplier * alpha_c
    kz_vacuum = k0_value * np.sin(alpha_i_value)
    kz_internal = k0_value * np.sqrt(
        np.sin(alpha_i_value) ** 2 - np.sin(alpha_c) ** 2
    )
    predicted = bulk_repeat * (kz_vacuum - kz_internal) / (2.0 * np.pi)
    color = mpl.colormaps["plasma"](
        column / max(len(bragg_multipliers) - 1, 1)
    )
    axis.semilogy(
        bragg_L, amplitude, color=color, linewidth=1.2,
        label=(
            rf"DWBA {multiplier:g}$\,\alpha_c$: "
            rf"$\Delta L={peak_L - 2.0:+.4f}$ "
            rf"(predicted {predicted:+.4f})"
        ),
    )
    axis.axvline(peak_L, color=color, linewidth=0.8, linestyle=":")
axis.axvline(2.0, color="0.35", linewidth=0.9, linestyle=":")
axis.set_xlabel(r"$L$ / r.l.u.")
axis.set_ylabel(r"$|F|$ / electrons")
axis.set_title("Refraction shift of the (0, 1, 2) bulk Bragg pole")
axis.grid(alpha=0.18, which="both")
axis.legend(fontsize=8)
figure

The measured peak positions reproduce the predicted shift to the fourth
decimal, so the Bragg-pole dips in the ratio plot are the refraction shift and
nothing else. They are real, but they are a shift in $L$, not a change of
intensity, and they shrink as $\alpha_i^{-1}$.

Note that this pole shift, $0.0082$ r.l.u. at $1.5\,\alpha_c$, is smaller than
the $0.0141$ r.l.u. used for the rod between the poles. The pole is carried by
the semi-infinite TiO2 bulk and follows the substrate $\delta$, while the rod
between the poles is carried by the RuO2 film and interface records and follows
the larger film $\delta$. `refraction_L_shift` takes the medium as an argument
for exactly this reason.

## Conclusion

The high-$L$ non-specular disagreement reported for the Poisson dissolution
family is real in the sense that the two curves do differ, but it is not mostly
a distorted-wave effect. Ordered by size at $(0,1,3)$ with four dissolved
layers:

| term | size here | physical? |
| --- | --- | --- |
| bulk truncation, `atten = 0.01` versus DWBA $\mathrm{Im}\,k_z$ | $\times 1.73$ | no, a convention mismatch |
| optical field $\lvert A_i^{+}A_f^{+}
vert$ in the RuO2 film, plus reflected channels | $\times 1.1$ to $1.3$ | yes |
| polarization contraction $\cos\varphi$ | $\times 0.978$ | yes, but absent from `SXRDCrystal.F` |
| refraction shift of $Q_z$ | $\Delta L = 0.0082$ (bulk pole), $0.0141$ (film) | yes; decisive at any sharp feature |

The truncation term is invisible on a smooth surface and on the specular rod,
which is why the reference notebook sees agreement there. It becomes dominant
precisely where the deeply dissolved non-specular rods have a near-cancellation
between the bulk term and the surface correction, and it grows with the amount
of dissolved material because the cancellation deepens. That is the reported
"more than four dissolved monolayers" signature.

Practical guidance that follows:

* compare DWBA against a kinematical rod whose `atten` comes from
  `CTRutil.attenuation_from_dwba` at the same angles, not against a rod carrying
  a fitted or serialized empirical value;
* expect a residual $\cos\varphi$ of a few percent, because `SXRDCrystal.F` is a
  bare structure factor;
* choose the incidence angle against the critical angle of the *densest* medium
  in the stack, not the substrate. Here the film critical angle is
  $0.143^{\circ}$ against the substrate's $0.116^{\circ}$, so the nominally safe
  $1.5\,\alpha_c$ is really $1.21\,\alpha_c$ of the layer that carries the
  signal;
* correct the nominal $l$ for refraction, with the $\delta$ of the medium that
  carries the feature, before comparing anything narrow: a Bragg pole, a film
  thickness fringe, or a rod minimum. The shift is small in absolute terms and
  large in relative terms exactly where the amplitude is small.